# Honeypot data analysis

An ad-hoc exploration notebook that reads straight from the honeypot database
using the same query layer the API uses. Run the seeder first if the database
is empty:

```bash
python -m tools.seed_fake_data --attackers 150 --days 14
```

This notebook has **no plotting dependencies** — it uses pandas if present and
falls back to plain text otherwise, so it runs in a bare environment.

> Reminder: if you point this at a real deployment's database, everything in
> [FINDINGS.md](../FINDINGS.md) about handling captured credentials and source
> addresses applies.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from storage import queries
from storage.db import session_scope

try:
    import pandas as pd

    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas not installed - falling back to plain output")


def show(rows):
    if HAVE_PANDAS:
        return pd.DataFrame(rows)
    for r in rows:
        print(r)
    return None

## Headline numbers (last 14 days)

In [ ]:
WINDOW_H = 14 * 24
with session_scope() as db:
    summary = queries.summary_stats(db, since_hours=WINDOW_H)
summary

## Top credentials

The most-tried username/password pairs.

In [ ]:
with session_scope() as db:
    creds = queries.credential_pairs(db, limit=20, since_hours=WINDOW_H)
show(creds)

## Where it comes from

In [ ]:
with session_scope() as db:
    countries = queries.top_countries(db, limit=15, since_hours=WINDOW_H)
    asns = queries.top_asns(db, limit=10, since_hours=WINDOW_H)
print("Top countries:")
print(show(countries))
print("\nTop networks:")
show(asns)

## The attack schedule

Counts by weekday and UTC hour. A campaign clustered on a human schedule looks
different from one on a cron job - one of the more reliable ways to tell
hands-on-keyboard activity from automation.

In [ ]:
with session_scope() as db:
    hm = queries.hour_weekday_heatmap(db, since_hours=WINDOW_H)

if HAVE_PANDAS:
    grid = pd.DataFrame(hm["grid"], index=hm["weekdays"], columns=[f"{h:02d}" for h in range(24)])
    try:
        display(grid.style.background_gradient(cmap="Reds", axis=None).format("{:d}"))
    except Exception:
        display(grid)
else:
    for day, row in zip(hm["weekdays"], hm["grid"], strict=False):
        bar = "".join("#" if c > hm["max"] * 0.5 else ("." if c else " ") for c in row)
        print(f"{day} |{bar}|")

## Highest-threat sources

Ranked by the scoring model. Pull one apart with the score explainer to see
*why* it scored what it did.

In [ ]:
with session_scope() as db:
    attackers, _ = queries.list_attackers(db, limit=15, sort="threat_score")
    rows = [
        {
            "src_ip": a.src_ip,
            "score": a.threat_score,
            "class": a.classification,
            "country": a.country,
            "events": a.event_count,
            "services": ",".join(a.services or []),
        }
        for a in attackers
    ]
show(rows)

In [ ]:
# Explain the top-scoring source, component by component.
from sqlalchemy import select

from pipeline.detection.scoring import explain
from storage.models import Event

with session_scope() as db:
    top = attackers[0]
    events = list(db.execute(select(Event).where(Event.src_ip == top.src_ip)).scalars())
    breakdown = explain(top, events)

print(f"{top.src_ip}  -  score {breakdown['score']}  ({breakdown['classification']})")
print(f"raw {breakdown['raw_score']} x recency {breakdown['recency_decay']}\n")
for name, value in breakdown["components"].items():
    weight = breakdown["weights"][name]
    filled = int((value / weight) * 24) if weight else 0
    print(f"  {name:<20} {'#' * filled}{'.' * (24 - filled)}  {value:>5.1f}/{weight}")

## Second-stage payload URLs

URLs attackers asked the fake shell to fetch. **The sensor never requested
these** - they are recorded as intelligence about where the next-stage payloads
live. Do not fetch them from an ordinary machine.

In [ ]:
import datetime as dt

from pipeline.reporting.daily_summary import _collect_payload_urls
from storage.models import utcnow

with session_scope() as db:
    urls = _collect_payload_urls(db, utcnow() - dt.timedelta(hours=WINDOW_H))
print(f"{len(urls)} distinct payload URL(s) observed:")
for u in urls:
    print("  ", u.replace("http", "hxxp"))  # defanged for display